# NCPA estimation from recorded PSFs

This notebook fits a static DM command that cancels a non-common-path aberration (NCPA) -- a static wavefront error present only in the science PSF, invisible to the WFS -- from a matrix `C` of `n_iterations` known DM diversity commands and their recorded PSFs. See `Ideas/06-ncpa-estimation.md` for the full design rationale; this notebook follows that plan's numbered steps directly.

**The core idea:** `DeformableMirror.forward` is exactly linear in its input command (`dm(C_i + c) = dm(C_i) + dm(c)`), so a single static command `c_ncpa` that explains every recorded PSF via `dm(C_i + c_ncpa)` is *directly* the DM offset to apply going forward -- no sign flip, no extra conversion. Since there is no real bench behind this notebook, the "recorded" PSFs are synthesized by the exact same simulator under a known, injected ground-truth `c_ncpa` (and, optionally, a DM rotation/scaling drift) -- a self-consistency check in place of a dedicated pytest test, per this plan's discussion in `Ideas/06-ncpa-estimation.md`.

**Optional step:** because the science path can reach the DM through different relay optics than the WFS path `TwinCalibrator` calibrates against, this notebook also demonstrates jointly fitting `dm.rotationAngle`/`dm.radialScaling`/`dm.tangentialScaling` alongside `c_ncpa` (toggled by `NCPAParams["FitDMGeometry"]`), while every other already-calibrated DM property is left untouched.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from IPython.display import display, clear_output

from AI4AO import WFS, DeformableMirror, imshow_multiple, imshow

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Loading the configuration

`NCPAEstimation_params.py` holds `WFSParams` (telescope aperture -- only `wfs.pupil`/`D`/`Nres`/`wavelength` are ever used here, since `WFS.GetPSF` doesn't touch the WFS mask/sensing path at all), `DMParams` (the DM whose misregistration is "previously calibrated" per this plan), and `NCPAParams` (the fit itself: science wavelength, `GetPSF` sampling/FOV, diversity-command design, learning rates, and the optional-geometry-fit toggle). Unlike the other `Tutorials/Advanced` params files there is no `AtmosParams`/`LoopParams`/`TrainParams` -- NCPA is measured on a static internal source with zero atmosphere, so `PhaseDataset`/`Trainer` play no role anywhere in this notebook.

In [ ]:
paramfile = 'NCPAEstimation_params.py'

WFSParams = Config.fromfile(paramfile)['WFSParams']
DMParams = Config.fromfile(paramfile)['DMParams']
NCPAParams = Config.fromfile(paramfile)['NCPAParams']

wfs = WFS(WFSParams, device)

## Two DM instances: the "real bench" and the model being fit

There is no real bench behind this notebook, so `dm_true` plays that role: it represents the physical DM with its actual (to the fit, unknown) NCPA-inducing state, and is used only to synthesize "recorded" PSFs below -- it is frozen (`.eval()` + `requires_grad_(False)`) and never touched by any optimizer, the same two-DM pattern `Ideas/02-online-self-recalibrating-dm.md` uses for its own drift-recovery check. `dm_model` is the twin this notebook actually fits, left at nominal (zero) misregistration -- i.e. exactly what `TwinCalibrator` would already have on file for the WFS-path calibration.

`simulate_psfs` is the one forward model shared by every step below: `OPD = dm(commands)` -> `PSF = wfs.GetPSF(OPD, ...)`, normalized to unit flux (this plan's chosen photometric handling -- see `Ideas/06-ncpa-estimation.md`'s "Photometry" discussion). It is used both to synthesize the ground-truth "recorded" data (with `dm_true`) and inside the fit loop (with `dm_model`).

In [ ]:
dm_true = DeformableMirror(WFSParams, DMParams, device)
dm_true.eval()

dm_model = DeformableMirror(WFSParams, DMParams, device)

nacts = int(dm_model.totalAct.item())


def simulate_psfs(dm, commands, wl, sampling, fov):
    """OPD -> PSF forward model shared by the synthetic 'recorded' data and the fit itself.
    Normalizes to unit flux (this plan's chosen photometric handling)."""
    opd = dm(commands)
    psf = wfs.GetPSF(opd, sampling=sampling, fov=fov, wl=wl)
    return psf / psf.sum(dim=(-2, -1), keepdim=True)


print(f"Total actuators: {nacts}")

## Diversity command matrix `C`

Following step 7 of the plan: `C` is built from a handful of low-order Zernike modal pokes (converted to zonal actuator commands via `dm_model.MakeZernikeM2C`) rather than random zonal noise, mirroring the classical phase-diversity literature's use of a known, interpretable diversity phase. `M2C`'s columns follow `Zernike`'s Noll ordering (`AI4AO/PhaseDataset.py:Zernike`, `zernIndex(i+1)` for column `i`): columns 0-1 are pure tip/tilt (deliberately skipped -- the science path can't distinguish those from a PSF centroid shift), column 2 is focus, columns 3-4 are astigmatism, columns 5-6 are coma. Including an astigmatism/coma poke (not just focus) matters if the optional geometry fit is enabled below: a purely rotationally-symmetric diversity set (focus alone) cannot make `rotationAngle` identifiable at all.

**Units matter here and are easy to get wrong:** `NCPAParams["DiversityAmplitudes"]` are *dimensionless* `MakeZernikeM2C` coefficients, not meters -- `MakeZernikeM2C`'s `z` already carries a `1/wavenumber` OPD-per-unit-coefficient scaling internally (`DeformableMirror.py:230`), so a coefficient of `1.0` corresponds to a large physical OPD, and passing a meters-scale number directly (an early version of this notebook used `3e-8`/`8e-8`, intending 30/80nm) produces diversity pokes with essentially zero actual OPD -- confirmed directly below, and the resulting degenerate fit doesn't necessarily look obviously broken (it can still converge to *a* solution, just not a meaningful one). `NCPAParams["DiversityAmplitudes"] = [0.3, 0.7]` was chosen by directly checking `dm_model(C)`'s OPD RMS lands at an NCPA-scale ~30-80nm, not by guessing.

In [ ]:
nModes = int(nacts * 0.7)
M2C = dm_model.MakeZernikeM2C(nModes=nModes)  # (nacts, nModes)

diversity_columns = NCPAParams["DiversityModeColumns"]
diversity_amplitudes = NCPAParams["DiversityAmplitudes"]

z_diversity_rows = [torch.zeros(nModes, device=device)]  # a zero/reference command, always included
for col in diversity_columns:
    for amp in diversity_amplitudes:
        z = torch.zeros(nModes, device=device)
        z[col] = amp
        z_diversity_rows.append(z)

z_diversity = torch.stack(z_diversity_rows)  # (n_iterations, nModes)
C = z_diversity @ M2C.T                       # (n_iterations, nacts)
n_iterations = C.shape[0]

with torch.no_grad():
    opd_rms_per_row = dm_model(C)[..., dm_model.pupil.bool()].pow(2).mean(dim=-1).sqrt()
print(f"n_iterations = {n_iterations}, nModes = {nModes}, nacts = {nacts}")
print(f"Per-command OPD RMS (m): min={opd_rms_per_row.min():.2e}, max={opd_rms_per_row.max():.2e}"
      " -- sanity check that the diversity pokes are NCPA-scale, not accidentally ~0")

## Synthetic "recorded" PSFs (step 9's self-consistency check)

`c_ncpa_true` and (if `NCPAParams["FitDMGeometry"]`) the injected `dm_true.rotationAngle`/`radialScaling`/`tangentialScaling` are ground-truth values used only to synthesize this notebook's stand-in "recorded" data -- **not** measured from any real instrument (per `CLAUDE.md`'s rule against inventing real hardware parameter values). A real run of this notebook would instead load actual recorded PSFs from disk in place of this cell, with `dm_true` never constructed at all.

Note `dm_true.rotationAngle = ...` only updates the underlying parameter -- it does not by itself rebuild `self.IF` (unlike, e.g., `flip_lr`'s setter). `dm_true.MakeZonalModes()` must be called explicitly afterward so the injected drift actually shows up in `dm_true`'s propagated OPD.

In [ ]:
torch.manual_seed(0)

# A handful of actuators poked at +/-50nm -- a modest, DM-representable static aberration.
c_ncpa_true = torch.zeros(nacts, device=device)
poked = torch.randperm(nacts, device=device)[:nacts//2]
c_ncpa_true[poked] = 0.03 * (2 * torch.rand(nacts//2, device=device) - 1)
c_ncpa_true += 0.02 * (2 * torch.rand(nacts, device=device) - 1)

if NCPAParams["FitDMGeometry"]:
    with torch.no_grad():
        dm_true.rotationAngle = torch.tensor([6.0], device=device)         # degrees
        dm_true.radialScaling = torch.tensor([0.02], device=device)        # fraction
        dm_true.tangentialScaling = torch.tensor([-0.015], device=device)  # fraction
        dm_true.MakeZonalModes()  # rebuild self.IF so the injected drift takes effect

with torch.no_grad():
    recorded_psfs = simulate_psfs(
        dm_true, C + c_ncpa_true.unsqueeze(0),
        NCPAParams["ScienceWavelength"], NCPAParams["PSFSampling"], NCPAParams["PSFFov"],
    )

print(f"recorded_psfs shape: {tuple(recorded_psfs.shape)}")
imshow(recorded_psfs)
plt.show()

## The fit

`fit_ncpa` implements steps 2-6 (and, when `fit_geometry=True`, step 8) in one function. The two branches freeze the DM's misregistration differently, and getting this right is the plan's central correctness trap either way:

- **`fit_geometry=False` (step 2):** `dm.eval()` freezes every misregistration parameter and rebuilds `self.IF` once under `torch.no_grad()` -- correct here, since only `c_ncpa` should move.
- **`fit_geometry=True` (step 8):** `dm.train()` is required instead, so `DeformableMirror.forward`'s `if self.training: self.MakeZonalModes()` rebuilds `self.IF` *inside* the autograd graph on every call, letting gradients reach `rotationAngle`/`radialScaling`/`tangentialScaling`. But `dm.train()` sets `requires_grad_(True)` on *every* DM parameter indiscriminately, so the ones that "should not have changed" (`grid_shift`, `sign`, `moffatParameter`, `mechCoupling`, `anamorphosisAngle`) are explicitly re-frozen right after.

`c_ncpa` itself also needs the same optimizer-friendly reparametrization `CLAUDE.md`'s conventions call for elsewhere in this codebase (e.g. `DeformableMirror.sign`'s `1e-6` scale): the physical command is a handful of tens-of-nm-scale actuator strokes (~1e-8 m), and Adam's per-step update size is roughly `lr` regardless of a raw parameter's own scale -- an early version of this notebook used an unscaled `c_ncpa` with `lr=3e-2` and, as verified in a headless smoke test of this notebook's code, that alone moved the fit by *centimeters* per step, completely swamping the actual ~50nm signal. `fit_ncpa` instead optimizes a dimensionless `c_ncpa_raw` at `NCPA_COMMAND_SCALE = 1e-6` -- matching `DeformableMirror.sign`'s own scale exactly, since both represent actuator stroke amplitudes in meters -- and returns the physical, meters-valued command (`c_ncpa_raw * NCPA_COMMAND_SCALE`), so nothing downstream needs to know about the internal reparametrization.

Either way, `dm.eval()` is called again before returning, so `dm_model` is left in a clean, deterministic, frozen state (with whatever geometry was fit baked into `self.IF`) for the validation plots below. The live plot mirrors `TwinCalibrator.fit_dm_and_offsets`'s use of `imshow_multiple` for its own residual-image tracking.

In [ ]:
NCPA_COMMAND_SCALE = 1e-2  # matches DeformableMirror.sign's own reparametrization scale


def fit_ncpa(dm, C, recorded_psfs, ncpa_params, fit_geometry=False, live_plot=True, number_to_show = 4):
    """Fits c_ncpa (and, if fit_geometry, dm's rotation/radial/tangential scaling) so that
    dm(C + c_ncpa), propagated through wfs.GetPSF, matches recorded_psfs. See
    Ideas/06-ncpa-estimation.md steps 2-6 and 8 for the full design rationale. Returns the
    physical, meters-valued c_ncpa -- internally it optimizes a dimensionless c_ncpa_raw at
    NCPA_COMMAND_SCALE, since Adam's per-step update size is roughly `lr` regardless of a raw
    parameter's own physical scale (see the markdown cell above)."""
    nacts_ = int(dm.totalAct.item())
    c_ncpa_raw = torch.nn.Parameter(torch.zeros(nacts_, device=device))

    if fit_geometry:
        dm.train()
        for p in (dm._grid_shift, dm._sign, dm._moffatParameter, dm._mechCoupling, dm._anamorphosisAngle):
            p.requires_grad_(False)
        param_groups = [
            {"params": [dm._rotationAngle, dm._radialScaling, dm._tangentialScaling],
             "lr": ncpa_params["LearningRateGeometry"]},
            {"params": [c_ncpa_raw], "lr": ncpa_params["LearningRateNCPA"]},
        ]
    else:
        dm.eval()
        param_groups = [{"params": [c_ncpa_raw], "lr": ncpa_params["LearningRateNCPA"]}]

    optimizer = torch.optim.AdamW(param_groups, fused=(device == 'cuda'))
    reg_weight = ncpa_params["RegularizationWeight"]
    wl = ncpa_params["ScienceWavelength"]
    sampling = ncpa_params["PSFSampling"]
    fov = ncpa_params["PSFFov"]
    n_iter = ncpa_params["NIterations"]

    if live_plot:
        zero_diff = torch.zeros_like(recorded_psfs)
        fig, ax = imshow_multiple([
            {"tensor": recorded_psfs, "title": "Recorded PSF (iter 0)", "same_scale": True},
            {"tensor": recorded_psfs, "title": "Simulated PSF (iter 0)", "same_scale": True},
            {"tensor": zero_diff, "title": "Difference"},
        ], max_channel_number = number_to_show
        )

    loss_tracker = torch.zeros(n_iter)
    progressBar = tqdm(range(n_iter))
    for u in progressBar:
        optimizer.zero_grad(set_to_none=True)

        c_ncpa = c_ncpa_raw * NCPA_COMMAND_SCALE
        psf_sim = simulate_psfs(dm, C + c_ncpa.unsqueeze(0), wl, sampling, fov)
        l = ((psf_sim - recorded_psfs) ** 2).sum() + reg_weight * (c_ncpa_raw ** 2).sum()
        l.backward()
        optimizer.step()

        loss_tracker[u] = l.detach()
        progressBar.set_postfix({"loss": float(l)})

        if live_plot and u % 10 == 0:
            clear_output(wait=True)
            imshow_multiple([
                {"tensor": recorded_psfs, "title": "Recorded PSF (iter 0)", "same_scale": True},
                {"tensor": psf_sim.detach(), "title": "Simulated PSF (iter 0)", "same_scale": True},
                {"tensor": (recorded_psfs - psf_sim).detach(), "title": "Difference"},
            ], fig=fig, axes=ax, max_channel_number = number_to_show)
            display(fig, clear=True)
            plt.pause(0.1)

    if live_plot:
        plt.close(fig)

    dm.eval()  # leave dm in a clean, deterministic, frozen state (whatever geometry was fit is now baked into self.IF)
    return (c_ncpa_raw.detach() * NCPA_COMMAND_SCALE), loss_tracker

## Running the baseline fit (`c_ncpa` only)

Always run first, regardless of `NCPAParams["FitDMGeometry"]` -- if a geometry drift was injected above, this baseline (geometry-blind) fit is expected to do visibly worse, demonstrating the problem the optional step 8 solves.

In [ ]:
c_ncpa_baseline, loss_baseline = fit_ncpa(dm_model, C, recorded_psfs, NCPAParams, fit_geometry=False)

print(f"Baseline final loss: {loss_baseline[-1]:.3e}")

## Recovering the correction command (step 6)

`dm_model.forward` is exactly linear in its input command (no misregistration is being fit in the baseline branch), so `c_ncpa_baseline` should already satisfy `dm(C_i + c_ncpa) = dm(C_i) + dm(c_ncpa)` for any `C_i` -- verified directly below, rather than only asserted in the plan doc.

In [ ]:
with torch.no_grad():
    lhs = dm_model(C[:1] + c_ncpa_baseline.unsqueeze(0))
    rhs = dm_model(C[:1]) + dm_model(c_ncpa_baseline.unsqueeze(0))
    assert torch.allclose(lhs, rhs, atol=1e-12), "dm.forward is expected to be exactly linear in its command input"

print("Linearity check passed: c_ncpa_baseline is directly usable as a DM offset command.")

## Validating recovery (step 9)

A low final loss is not by itself evidence of correct recovery -- the actuator-level comparison below is the actual bar to clear. If `NCPAParams["FitDMGeometry"]` is `False` (no drift was injected), the baseline fit should recover `c_ncpa_true` closely; if it is `True`, this geometry-blind fit is expected to show a visibly imperfect recovery -- the geometry-aware fit further below should do better.

In [ ]:
with torch.no_grad():
    true_opd = dm_true(c_ncpa_true.unsqueeze(0))[0]
    fitted_opd = dm_model(c_ncpa_baseline.unsqueeze(0))[0]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(c_ncpa_true.cpu(), c_ncpa_baseline.cpu())
lim = float(c_ncpa_true.abs().max()) * 1.2
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1)
ax.set_xlabel("Injected c_ncpa (m)")
ax.set_ylabel("Fitted c_ncpa (m)")
ax.set_title("Baseline fit: per-actuator recovery")
plt.show()

imshow_multiple([
    {"tensor": true_opd, "title": "True NCPA OPD", "same_scale": True},
    {"tensor": fitted_opd, "title": "Fitted NCPA OPD (baseline)", "same_scale": True},
    {"tensor": true_opd - fitted_opd, "title": "Difference", "scale_reference": true_opd},
])
plt.show()

## Optional step 8: jointly calibrating DM rotation and radial/tangential scaling

Only meaningful when `NCPAParams["FitDMGeometry"]` is `True` (both the synthetic ground truth above and this fit are gated by the same toggle -- see the "Synthetic 'recorded' PSFs" cell). In a real (non-synthetic) use of this notebook you would not know ahead of time whether the science path's relay optics rotate or magnify the DM; this toggle only controls whether *this demo* simulates that and, if so, asks `fit_ncpa` to compensate for it.

Because `dm_model` was left in `dm.eval()` mode by the baseline fit above, its `rotationAngle`/`radialScaling`/`tangentialScaling` are still at their nominal (zero) starting point -- reusing the same `dm_model` instance here is safe, nothing needs to be reset.

In [ ]:
if NCPAParams["FitDMGeometry"]:
    c_ncpa_geom, loss_geom = fit_ncpa(dm_model, C, recorded_psfs, NCPAParams, fit_geometry=True)

    print(f"Baseline (geometry-blind) final loss: {loss_baseline[-1]:.3e}")
    print(f"Geometry-aware final loss:            {loss_geom[-1]:.3e}")
    print()
    print(f"rotationAngle      -- true: {dm_true.rotationAngle.item():+.3f} deg, "
          f"fitted: {dm_model.rotationAngle.item():+.3f} deg")
    print(f"radialScaling      -- true: {dm_true.radialScaling.item():+.4f},     "
          f"fitted: {dm_model.radialScaling.item():+.4f}")
    print(f"tangentialScaling  -- true: {dm_true.tangentialScaling.item():+.4f},     "
          f"fitted: {dm_model.tangentialScaling.item():+.4f}")
else:
    print("NCPAParams['FitDMGeometry'] is False -- skipping the optional geometry fit "
          "(no drift was injected into dm_true either, so there is nothing to recover).")

In [ ]:
if NCPAParams["FitDMGeometry"]:
    with torch.no_grad():
        fitted_opd_geom = dm_model(c_ncpa_geom.unsqueeze(0))[0]

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(c_ncpa_true.cpu(), c_ncpa_baseline.cpu(), label="Baseline (geometry-blind)", alpha=0.6)
    ax.scatter(c_ncpa_true.cpu(), c_ncpa_geom.cpu(), label="Geometry-aware", alpha=0.6)
    lim = float(c_ncpa_true.abs().max()) * 1.2
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1)
    ax.set_xlabel("Injected c_ncpa (m)")
    ax.set_ylabel("Fitted c_ncpa (m)")
    ax.set_title("Per-actuator recovery: baseline vs. geometry-aware")
    ax.legend()
    plt.show()

    imshow_multiple([
        {"tensor": true_opd, "title": "True NCPA OPD", "same_scale": True},
        {"tensor": fitted_opd, "title": "Fitted OPD (baseline)", "same_scale": True},
        {"tensor": fitted_opd_geom, "title": "Fitted OPD (geometry-aware)", "same_scale": True},
        {"tensor": true_opd - fitted_opd_geom, "title": "Difference (geometry-aware)", "scale_reference": true_opd},
    ])
    plt.show()